# Phase 6: preregistered generic controls

This notebook analyzes the frozen paired-control experiment: three local and three broad generic-real symplectic deformations for each of 4,165 CM candidates. The primary unit of analysis is the CM candidate, after averaging its three controls.

In [1]:
import gzip, json, math, statistics
from collections import Counter, defaultdict
from pathlib import Path

here = Path.cwd().resolve()
candidates = (here, here.parent, here / 'passive-cliffords')
project = next(path for path in candidates if (path / 'src' / 'gkp_passive_cliffords').exists())
data = project / 'data'
def load_json(name):
    path = data / name
    if path.exists(): return json.loads(path.read_text())
    with gzip.open(str(path) + '.gz', 'rt') as handle: return json.load(handle)
rows = load_json('phase6_generic_controls.json')
summaries = load_json('phase6_generic_controls_summary.json')
protocol = load_json('phase6_preregistered_protocol.json')
audits_60d = load_json('phase6_high_precision_audit.json')
print(protocol['protocol_version'])
print(f"Loaded {len(rows):,} controls for {len({r['candidate_id'] for r in rows}):,} CM candidates.")
print(protocol['regimes'])

phase6-v1-preregistered
Loaded 24,990 controls for 4,165 CM candidates.
[{'name': 'local', 'samples_per_candidate': 3, 'amplitude': 0.002, 'steps': 4, 'vector_bound': 2, 'coefficient_denominator': 10000000}, {'name': 'broad', 'samples_per_candidate': 3, 'amplitude': 0.05, 'steps': 4, 'vector_bound': 1, 'coefficient_denominator': 10000000}]


## Candidate-level paired distance analysis

We report $\Delta\ell^2=\ell^2_{\rm control}-\ell^2_{\rm CM}$, so negative values favor the CM baseline. The interval is a descriptive 95% normal interval across candidate-level paired differences.

In [2]:
def paired_stats(control_rows):
    by_candidate = defaultdict(list)
    for row in control_rows:
        by_candidate[row['candidate_id']].append(row)
    differences, ratios, wins = [], [], []
    for candidate_rows in by_candidate.values():
        cm = candidate_rows[0]['cm_ell_squared']
        control = statistics.mean(r['control_ell_squared'] for r in candidate_rows)
        differences.append(control - cm)
        ratios.append(control / cm)
        wins.append(control > cm)
    average = statistics.mean(differences)
    standard_error = statistics.stdev(differences) / math.sqrt(len(differences))
    return {
        'n': len(differences), 'mean_difference': average,
        'ci_low': average - 1.96 * standard_error,
        'ci_high': average + 1.96 * standard_error,
        'mean_ratio': statistics.mean(ratios),
        'median_ratio': statistics.median(ratios),
        'control_mean_beats_cm_fraction': statistics.mean(wins),
    }

print('type | regime | candidates | mean control-CM | 95% interval | mean ratio | control mean beats CM')
print('--- | --- | ---: | ---: | ---: | ---: | ---:')
types = sorted({tuple(row['polarization_type']) for row in rows})
paired = {}
for D in types:
    for regime in ('local', 'broad'):
        selected = [r for r in rows if tuple(r['polarization_type']) == D and r['regime'] == regime]
        result = paired_stats(selected)
        paired[(D, regime)] = result
        interval = f"[{result['ci_low']:.6f}, {result['ci_high']:.6f}]"
        print(f"{D} | {regime} | {result['n']} | {result['mean_difference']:.6f} | {interval} | {result['mean_ratio']:.6f} | {result['control_mean_beats_cm_fraction']:.6f}")

type | regime | candidates | mean control-CM | 95% interval | mean ratio | control mean beats CM
--- | --- | ---: | ---: | ---: | ---: | ---:
(1, 1, 2) | local | 1051 | -0.005318 | [-0.006103, -0.004534] | 0.993520 | 0.392008
(1, 1, 2) | broad | 1051 | -0.005950 | [-0.011093, -0.000806] | 1.044508 | 0.594672
(1, 1, 3) | local | 1070 | -0.003865 | [-0.004529, -0.003201] | 0.994355 | 0.393458
(1, 1, 3) | broad | 1070 | 0.002515 | [-0.002175, 0.007205] | 1.076701 | 0.606542
(1, 2, 2) | local | 253 | -0.005397 | [-0.006673, -0.004120] | 0.989711 | 0.245059
(1, 2, 2) | broad | 253 | -0.006037 | [-0.015387, 0.003313] | 1.039141 | 0.545455
(1, 3) | local | 876 | -0.026669 | [-0.029882, -0.023457] | 0.953813 | 0.393836
(1, 3) | broad | 876 | -0.056890 | [-0.066016, -0.047764] | 0.999359 | 0.469178
(1, 5) | local | 915 | -0.009892 | [-0.011767, -0.008017] | 0.980396 | 0.567213
(1, 5) | broad | 915 | -0.024643 | [-0.031178, -0.018108] | 1.145540 | 0.497268


In [3]:
for regime in ('local', 'broad'):
    result = paired_stats([r for r in rows if r['regime'] == regime])
    print(regime, result)

local {'n': 4165, 'mean_difference': -0.010445108030341021, 'ci_low': -0.011321916952010829, 'ci_high': -0.009568299108671213, 'mean_ratio': 0.982268723888335, 'median_ratio': 0.9969677228843815, 'control_mean_beats_cm_fraction': 0.4223289315726291}


broad {'n': 4165, 'mean_difference': -0.018600990117969125, 'ci_low': -0.021705789362463798, 'ci_high': -0.015496190873474452, 'mean_ratio': 1.0651518967825109, 'median_ratio': 1.02380491377665, 'control_mean_beats_cm_fraction': 0.5469387755102041}


## Passive-gate audit

The outcome-independent audit selected 25 candidates per type and regime, always using control index zero. A fixed one-second exhaustive-enumeration cap labels difficult cases `bounded_unresolved`; it does not replace or resample them.

In [4]:
selected = [row for row in rows if row['gate_audited']]
certified = [row for row in selected if row['gate_audit_status'] == 'certified']
unresolved = [row for row in selected if row['gate_audit_status'] == 'bounded_unresolved']
print('selected:', len(selected), 'certified:', len(certified), 'bounded unresolved:', len(unresolved))
print('certified control image orders:', Counter(row['control_logical_image_order'] for row in certified))
print('controls with enhanced image:', sum(row['control_has_extra_passive_symmetry'] for row in certified))
print('paired CM image strictly larger:', sum(row['cm_image_exceeds_control'] for row in certified), '/', len(certified))

selected: 250 certified: 233 bounded unresolved: 17
certified control image orders: Counter({2: 134, 1: 99})
controls with enhanced image: 0
paired CM image strictly larger: 50 / 233


## Deformation scale and claim boundary

The regime labels fix generator coefficients, not actual metric distance. Long displacement tails—especially for broad controls based at ill-conditioned metrics—mean the local experiment is the cleaner comparison.

In [5]:
for regime in ('local', 'broad'):
    displacement = sorted(r['relative_metric_displacement'] for r in rows if r['regime'] == regime)
    q95 = displacement[int(0.95 * (len(displacement) - 1))]
    print(regime, 'median displacement =', statistics.median(displacement), 'q95 =', q95)

local median displacement = 0.0834960434479751 q95 = 0.762550765819402
broad median displacement = 0.8033175458477492 q95 = 84.97322236661329


## Reproducibility checks

In [6]:
expected = {(1,3): 876, (1,5): 915, (1,1,2): 1051, (1,1,3): 1070, (1,2,2): 253}
assert protocol['adaptive_resampling'] is False
assert len(rows) == 24_990
assert len({r['candidate_id'] for r in rows}) == 4_165
assert len(selected) == 250 and len(certified) == 233 and len(unresolved) == 17
assert all(not r['control_has_extra_passive_symmetry'] for r in certified)
assert all(paired[(D, 'local')]['mean_difference'] < 0 for D in expected)
assert len(audits_60d) == 10
assert max(a['absolute_difference'] for a in audits_60d) < 2e-12
print('Phase 6 preregistered-control checks passed.')

Phase 6 preregistered-control checks passed.
